# VoiceArm — planner ablation: does a Tamil-native 24B plan better?

[VoiceArm](https://github.com/dheepakkaran/VoiceArm-Bilingual-Tamil-English-Voice-Controlled-Robotic-Arm-MuJoCo) ships `Qwen3-4B` as its task planner. `sarvam-m` (24B) is the
strongest open model for Tamil and was rejected for one reason only: at 4-bit it
needs about 13 GB and will not co-reside with two ASR models and a detector on a
16 GB laptop.

That is a hardware constraint, not a quality claim. This notebook makes the
comparison the laptop could not, on eight reference utterances spanning English,
Tanglish and Tamil script.

No MuJoCo here -- the planner is a pure text-in, JSON-out stage, so this needs
nothing but transformers.

**Accelerator:** GPU. `T4 x2` (32 GB) is comfortable; a single 16 GB P100 works
but offloads a few layers to CPU and runs slower.

In [ ]:
import torch

print("gpu  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("count:", torch.cuda.device_count())
print("vram :", ", ".join(
    f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.0f} GB"
    for i in range(torch.cuda.device_count())) or "-")
print("bf16 :", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes 2>&1 | tail -2

import bitsandbytes
import transformers

print("transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJ = Path("/kaggle/working") / "voicearm"
if not PROJ.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dheepakkaran/VoiceArm-Bilingual-Tamil-English-Voice-Controlled-Robotic-Arm-MuJoCo.git", str(PROJ)], check=True)

os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
os.environ["VOICEARM_BACKEND"] = "torch"
print("project at", PROJ)

## The comparison

`scripts/bench_planner.py` scores each model on whether the first step targets
the right object and whether a `place` step appears exactly when the instruction
asks for one. It detects that the community `sarvam-m` upload is already
quantized and does not re-quantize it, and it picks float16 over bfloat16 on GPUs
without bf16 units -- a T4 is one of those.

Downloads about 22 GB: 8 GB for Qwen3-4B and 14 GB for the pre-quantized
sarvam-m, against 47 GB for the official fp16 weights.

In [ ]:
!python scripts/bench_planner.py     --models Qwen/Qwen3-4B-Instruct-2507 neuralnets/sarvam-m-4bit-q     --load-4bit

In [ ]:
import json
from pathlib import Path

results = Path("out/planner_bench.json")
if results.exists():
    rows = json.loads(results.read_text())
    print(f"{'model':<40}{'correct':>9}{'mean gen':>10}{'peak vram':>11}")
    for row in rows:
        print(f"{row['model']:<40}{row['correct']:>5}/{row['of']}"
              f"{row['mean_gen_s']:>9.2f}s{row.get('peak_vram_gb', float('nan')):>10.1f}G")
else:
    print("no results -- check the cell above for a load failure")

## Reading the result

If the 24B model does not score higher, the README's model choice is confirmed on
evidence rather than on footprint, which is a stronger claim than the one it
makes now.

If it does score higher, that is a real finding: the shipped planner is limited
by the laptop, and the constraint is worth stating as a measured cost.

Either way the latency and VRAM columns matter. A model that is more accurate but
six times slower is a different trade, not a free win.